# SpecDist -- Colab A100 Quickstart

**Train Qwen3-0.6B draft from Qwen3-8B teacher on a free A100 (Colab Pro/Pro+).**

**CONFIG levels:** `colab_lite` (1.7B teacher, ~25 min) -> `colab` (4B teacher, ~2-4 h, T4) -> `colab_a100` (8B teacher, ~2-3 h, **this notebook**)

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | **One-shot: setup + auth + run (recommended)** | starts in ~3 min |
| 1. Setup | Mount Drive, clone repo, install deps | ~3 min |
| 2. Authenticate | W&B + HuggingFace (from Colab Secrets) | ~30 s |
| 3. Run | Full pipeline (train -> merge -> eval) | ~2-3 h |
| 4. Resume | After session death -- re-runs from last checkpoint | ~1 min + training |
| 5. Monitor | Check progress log + state (auto-refresh option) | instant |
| 6. Dashboard | Live results dashboard in Colab tab | ~10 s |
| 7. Tree loss (single) | Train one tree loss (set `LOSS = "kl_tree"`, etc.) | ~10-30 min/loss |
| 8. Tree losses (all) | Train all 9 tree losses sequentially | ~2-3 h |

---

### Requires Colab Pro or Pro+
Runtime -> Change runtime type -> **A100 GPU**
(Standard free tier gives T4 only -- use `colab_quickstart.ipynb` for T4.)

### Quickest start
1. **Runtime -> Run all** (Ctrl+F9) -- Cell 0 handles everything automatically.
   Set `SMOKE = True` first for a quick 30-min smoke test.
2. To monitor while running: set `BACKGROUND = True` in Cell 0, then run Cell 5
   with `AUTO_REFRESH = True`.

### Why A100 over T4?
- Qwen3-8B teacher loads as **plain BF16** (no 4-bit quantization needed)
- VRAM budget: 1.2 GB draft + 16 GB teacher + 2-3 GB activations = ~19-20 GB
  (A100 has 40 GB -- 20 GB headroom)
- Full alpha token-acceptance metrics (both models in VRAM simultaneously)
- ~4x faster training than T4; 2000 steps in ~15-20 min vs ~30 min
- `torch.compile` enabled: additional 10-30% speedup
- Larger LoRA rank (r=16) for better adaptation across the 8B->0.6B gap

### Prerequisites
1. **A100 runtime**: Runtime -> Change runtime type -> A100 GPU
2. **Colab Secrets** (left sidebar -> 🔑 key icon):
   - `GITHUB_TOKEN` -- **required** (private repo). Create a classic PAT with `repo` scope at
     https://github.com/settings/tokens → "Generate new token (classic)" → tick **repo**
   - `WANDB_API_KEY` -- from https://wandb.ai/authorize
   - `HF_TOKEN` -- from https://huggingface.co/settings/tokens
3. Google Drive mounted (Cell 0 / Cell 1 does this automatically)

> **Note:** Qwen3-8B is ~16 GB. HuggingFace will download it on first run (~15 min).
> Subsequent runs use the cached copy in `/root/.cache/huggingface/`.

In [ ]:
# =============================================================================
# Cell 0 -- ONE-SHOT BOOTSTRAP for A100 (Colab Pro/Pro+)
#
# 1. Fill in the Config section below.
# 2. Runtime -> Run all (Ctrl+F9) to run every cell automatically.
#    OR run just this cell; it handles setup + auth + pipeline in one go.
#
# BACKGROUND = True  -> pipeline runs in background, cell exits immediately;
#                        run Cell 5 with AUTO_REFRESH=True to tail the log.
# BACKGROUND = False -> output streams directly to this cell (default).
#
# Teacher: Qwen3-8B (bf16, ~16 GB VRAM) -- A100 40 GB has plenty of room.
# VRAM budget: 1.2 (draft) + 16 (teacher) + 2-3 (KV/activations) = ~19-20 GB.
# Alpha eval works: both models in VRAM simultaneously.
# torch.compile enabled in colab_a100.yaml: first few steps slower (warm-up).
# WHY NOT 4-BIT?  Not needed -- A100 has 40 GB.  Also, 4-bit NF4 loading
# requires BF16 CPU-RAM intermediates (16 GB for 8B) -> SIGKILL on Colab.
#
# Session limit: Colab Pro ~12 h, Pro+ ~24 h. For T4 (free tier) use:
#   colab_quickstart.ipynb  (colab_lite: 1.7B, or colab: 4B teacher)
# For >24 h: use kaggle.ipynb or runpod launcher.
# =============================================================================

# -- Config (edit these) ------------------------------------------------------
DRIVE_ROOT  = "/content/drive/MyDrive/specdist"
CONFIG      = "colab_a100"  # colab_a100 | colab | colab_lite | server | laptop
SMOKE       = False      # True = smoke test (~45 min)    False = full run (~4 h)
BACKGROUND  = False      # True -> background process; monitor via Cell 5
START_DASHBOARD = False  # True -> also start Flask dashboard (Cell 6)
                         #         safe to set True even before any results exist
LOSSES      = None       # None = all losses, or e.g. "kl,ebe"
EXTRA_ARGS  = []         # e.g. ["--train_steps", "1000"]
REPO_URL    = "https://github.com/Rmuk655/Distill-Spec-Research.git"
# -----------------------------------------------------------------------------

import os, subprocess, sys, threading, time

# -- Keep-alive: prevent Colab idle-timeout while pipeline runs ---------------
from IPython.display import display, Javascript
display(Javascript("""
(function() {
    if (window.__specdist_keepalive) return;
    window.__specdist_keepalive = setInterval(function() {
        var evt = new MouseEvent('mousemove', {bubbles: true});
        document.dispatchEvent(evt);
        var btn = document.querySelector('[data-tooltip="Reconnect to runtime"]');
        if (btn) btn.click();
    }, 45000);
    console.log('[specdist] keep-alive started (45 s interval)');
})();
"""))

# -- Helper: read Colab Secret or env var -------------------------------------
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

# -- 1/5 Mount Drive ----------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(os.path.join(DRIVE_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "logs"),        exist_ok=True)
print(f"[1/5] Drive mounted  ->  {DRIVE_ROOT}")

# -- 2/5 Clone / update repo --------------------------------------------------
REPO_DIR = "/content/Distill-Spec-Research"
GBV_DIR  = f"{REPO_DIR}/gbv-research"
# Private repo: inject GitHub token into clone URL.
# Add GITHUB_TOKEN to Colab Secrets (left sidebar -> 🔑): classic PAT with repo scope.
# Create at: https://github.com/settings/tokens -> Generate new token (classic) -> tick repo
_gh = _secret("GITHUB_TOKEN")
if not _gh:
    print("[2/5] WARNING: GITHUB_TOKEN not set -- clone will fail for a private repo.")
    print("      Add it via: left sidebar -> 🔑 Secrets -> GITHUB_TOKEN")
_clone_url = REPO_URL.replace("https://", f"https://{_gh}@") if _gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", _clone_url, REPO_DIR], check=True)
    print("[2/5] Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("[2/5] Repo updated (was already cloned)")
os.chdir(GBV_DIR)

# -- 3/5 Install deps ---------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt", "bitsandbytes", "accelerate"], check=True)
print("[3/5] Dependencies installed")

# -- 4/5 Authenticate ---------------------------------------------------------
wb = _secret("WANDB_API_KEY")
if wb:
    os.environ["WANDB_API_KEY"] = wb
    import wandb; wandb.login(key=wb, relogin=True)
    print("[4/5] W&B authenticated")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("[4/5] W&B offline  (add WANDB_API_KEY via left sidebar -> Secrets)")

hf = _secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf
    try:
        from huggingface_hub import login
        login(token=hf, add_to_git_credential=False)
        print("     HuggingFace authenticated")
    except Exception: pass
else:
    print("     HF_TOKEN not set (OK for public Qwen3 models)")

import torch
if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"     GPU: {torch.cuda.get_device_name(0)}  ({free_gb:.1f} GB free)")
else:
    print("     WARNING: no GPU -- Runtime -> Change runtime type -> A100 GPU")

# -- 5/5 Run pipeline ---------------------------------------------------------
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      os.path.join(DRIVE_ROOT, "results.db"),
    "SPECDIST_LOGS_ROOT":    os.path.join(DRIVE_ROOT, "logs"),
})
LOG_FILE = os.path.join(DRIVE_ROOT, "logs", "pipeline_output.log")

cmd = [sys.executable, "orchestration/experiment.py",
       "--config", CONFIG, "--storage_root", DRIVE_ROOT, "--yes"]
if SMOKE:   cmd.append("--smoke")
if LOSSES:  cmd += ["--losses", LOSSES]
cmd += EXTRA_ARGS

mode_str = "SMOKE TEST" if SMOKE else f"FULL pipeline ({CONFIG})"
print(f"\n[5/5] Launching {mode_str}")
print(f"      Log  -> {LOG_FILE}")
print(f"      DB   -> {DRIVE_ROOT}/results.db")

if BACKGROUND:
    log_fh = open(LOG_FILE, "a", buffering=1)
    _proc  = subprocess.Popen(cmd, cwd=GBV_DIR, stdout=log_fh, stderr=subprocess.STDOUT)

    def _tail():
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as lf:
            lf.seek(0, 2)
            while _proc.poll() is None:
                line = lf.readline()
                if line: sys.stdout.write(line); sys.stdout.flush()
                else:    time.sleep(0.4)
            for line in lf: sys.stdout.write(line); sys.stdout.flush()

    threading.Thread(target=_tail, daemon=True).start()
    print(f"\n  Background PID: {_proc.pid}")
    print("  Output is tee'd to the Drive log above.")
    print("  Run Cell 5 now to monitor progress without blocking.")
    print(f"  To stop: import os, signal; os.kill({_proc.pid}, signal.SIGTERM)")
# -- Optional: start Flask dashboard in background ---------------------------
if START_DASHBOARD:
    import threading, time as _t
    from google.colab.output import eval_js as _ejs
    _DASH_PORT = 5000
    os.environ["SPECDIST_DB_PATH"]   = os.path.join(DRIVE_ROOT, "results.db")
    os.environ["SPECDIST_LOGS_ROOT"] = os.path.join(DRIVE_ROOT, "logs")
    _dash_proc = [None]

    def _start_dash():
        _dash_proc[0] = subprocess.Popen(
            [sys.executable,
             os.path.join(GBV_DIR, "dashboard", "training_dashboard.py"),
             "--host", "0.0.0.0", "--port", str(_DASH_PORT)],
            env=os.environ.copy())
        _dash_proc[0].wait()

    threading.Thread(target=_start_dash, daemon=True).start()
    _t.sleep(3)  # let Flask bind the port
    _dash_url = _ejs(f"google.colab.kernel.proxyPort({_DASH_PORT})")
    print(f"\nDashboard running -> {_dash_url}")
    print("  (auto-refreshes every 15 s; new eval rows appear as they land)")

# -- Launch pipeline ----------------------------------------------------------
else:
    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print("\n[DONE] Pipeline complete!  Run Cell 6 for the dashboard.")
    else:
        print(f"\n[FAIL] Exit code {result.returncode} -- re-run to resume from checkpoint.")
        print(f"       Full log -> {LOG_FILE}")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 1 -- Setup (A100)
# Mount Google Drive (ALL artifacts live here -- survive session restarts)
# Clone repo to ephemeral /content/ (fast, ~5 s; re-clones each session)
# Install dependencies
# -----------------------------------------------------------------------------
import os, subprocess, sys

# -- 1a. Mount Drive ----------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# storage_root is the single directory that holds all persistent artifacts:
#   results.db, checkpoints/, logs/, pipeline_state_*.json
DRIVE_ROOT = "/content/drive/MyDrive/specdist"
os.makedirs(os.path.join(DRIVE_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "logs"), exist_ok=True)
print(f"✓ All artifacts will persist at: {DRIVE_ROOT}")
print(f"    {DRIVE_ROOT}/results.db        ← experiment database")
print(f"    {DRIVE_ROOT}/checkpoints/      ← LoRA adapters")
print(f"    {DRIVE_ROOT}/logs/             ← pipeline + training logs")

# -- 1b. Clone repo -----------------------------------------------------------
REPO_URL  = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR  = "/content/Distill-Spec-Research"
GBV_DIR   = f"{REPO_DIR}/gbv-research"

# Private repo: read GitHub token from Colab Secrets.
# Add GITHUB_TOKEN via: left sidebar -> 🔑 Secrets -> GITHUB_TOKEN
# Create a classic PAT at https://github.com/settings/tokens (repo scope).
def _get_gh_token():
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t: return t
    except Exception: pass
    return os.environ.get("GITHUB_TOKEN", "")

_gh_tok    = _get_gh_token()
_clone_url = REPO_URL.replace("https://", f"https://{_gh_tok}@") if _gh_tok else REPO_URL
if not _gh_tok:
    print("⚠ GITHUB_TOKEN not set -- clone will fail for a private repo.")
    print("  Add it via: left sidebar -> 🔑 Secrets -> GITHUB_TOKEN")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", _clone_url, REPO_DIR], check=True)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated (already cloned)")

os.chdir(GBV_DIR)
print(f"✓ Working directory: {os.getcwd()}")

# -- 1c. Install Python dependencies -----------------------------------------
# bitsandbytes is required for 4-bit NF4 loading of the 8B teacher on a T4.
# accelerate is required by bitsandbytes device_map="auto".
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes",    # 4-bit NF4 teacher
    "accelerate",      # device_map support
    "flash-attn",      # optional -- faster attention; skipped silently if compile fails
], check=False)  # flash-attn may fail on older drivers -- that's OK
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes", "accelerate",
], check=True)   # re-run without flash-attn to ensure core deps are installed

print("✓ Dependencies installed")
print("\n--- Setup complete. Proceed to Cell 2 ---")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 2 -- Authenticate (W&B + HuggingFace)
#
# Reads secrets from Colab Secrets (left sidebar -> 🔑 icon).
# Required secrets:
#   WANDB_API_KEY  ->  https://wandb.ai/authorize
#   HF_TOKEN       ->  https://huggingface.co/settings/tokens
#
# If you prefer, paste keys directly below instead of using Secrets.
# -----------------------------------------------------------------------------
import os

def _get_secret(name: str, fallback: str = "") -> str:
    """Read from Colab Secrets, then env, then fallback."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name, fallback)

# -- W&B ----------------------------------------------------------------------
WANDB_API_KEY = _get_secret("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("✓ W&B authenticated")
else:
    print("⚠ WANDB_API_KEY not found -- runs will be logged offline only.")
    print("  Add it via: left sidebar -> 🔑 Secrets -> + Add new secret")
    os.environ["WANDB_MODE"] = "offline"

# -- HuggingFace --------------------------------------------------------------
HF_TOKEN = _get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN   # older env var
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✓ HuggingFace Hub authenticated")
    except Exception as e:
        print(f"  [HF login] {e} -- continuing anyway (public models don't need auth)")
else:
    print("ℹ HF_TOKEN not found -- OK for public models (Qwen3-0.6B, Qwen3-8B).")

# -- Confirm GPU ---------------------------------------------------------------
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}  "
          f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
    if total / 1024**3 < 12:
        print("⚠ Less than 12 GB VRAM detected -- the colab config requires T4 (15 GB).")
        print("  Runtime -> Change runtime type -> T4 GPU")
else:
    print("⚠ No GPU detected -- training will be extremely slow.")
    print("  Runtime -> Change runtime type -> T4 GPU")

print("\n--- Auth complete. Proceed to Cell 3 ---")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 3 -- Run the pipeline (A100)
#
# Prerequisite: Cells 1 + 2 must have run (or use Cell 0 instead).
# A100 config: Qwen3-8B teacher (bf16), lora_r=16, 2000 steps.
#
# BACKGROUND = False  -> output streams directly to this cell (default).
# BACKGROUND = True   -> pipeline runs as a background subprocess;
#                         this cell exits immediately so you can run
#                         Cell 5 with AUTO_REFRESH=True to monitor.
# -----------------------------------------------------------------------------
import os, subprocess, sys, threading, time

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# -- Configuration (edit these) -----------------------------------------------
CONFIG     = "colab_a100"  # colab_a100 | colab | colab_lite | server | laptop
SMOKE      = False      # True = quick smoke test (~45 min)
BACKGROUND = False      # True = background process; monitor with Cell 5
LOSSES     = None       # None = all, or "kl,ebe"
EXTRA_ARGS = []         # e.g. ["--train_steps", "1000"]
# -----------------------------------------------------------------------------

if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("DRIVE_ROOT not found -- run Cell 1 first.")

os.chdir(GBV_DIR)
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      os.path.join(DRIVE_ROOT, "results.db"),
    "SPECDIST_LOGS_ROOT":    os.path.join(DRIVE_ROOT, "logs"),
})
LOG_FILE = os.path.join(DRIVE_ROOT, "logs", "pipeline_output.log")

cmd = [sys.executable, "orchestration/experiment.py",
       "--config", CONFIG, "--storage_root", DRIVE_ROOT, "--yes"]
if SMOKE:   cmd.append("--smoke")
if LOSSES:  cmd += ["--losses", LOSSES]
cmd += EXTRA_ARGS

mode_str = "SMOKE TEST" if SMOKE else f"FULL pipeline ({CONFIG})"
print(f"{'[BG] ' if BACKGROUND else ''}{mode_str}")
print(f"  Log -> {LOG_FILE}")
print(f"  DB  -> {DRIVE_ROOT}/results.db")

if BACKGROUND:
    log_fh = open(LOG_FILE, "a", buffering=1)
    _proc  = subprocess.Popen(cmd, cwd=GBV_DIR, stdout=log_fh, stderr=subprocess.STDOUT)

    def _tail():
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as lf:
            lf.seek(0, 2)
            while _proc.poll() is None:
                line = lf.readline()
                if line: sys.stdout.write(line); sys.stdout.flush()
                else:    time.sleep(0.4)
            for line in lf: sys.stdout.write(line); sys.stdout.flush()

    threading.Thread(target=_tail, daemon=True).start()
    print(f"\nBackground PID {_proc.pid} -- run Cell 5 to monitor")
    print(f"To stop: import os, signal; os.kill({_proc.pid}, signal.SIGTERM)")
else:
    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print("\n[DONE] Pipeline complete!  Run Cell 6 for dashboard.")
    else:
        print(f"\n[FAIL] Exit code {result.returncode} -- re-run to resume.")
        print(f"       Full log -> {LOG_FILE}")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 4 -- Resume after session death
#
# Colab kills sessions after ~90 min idle (free) or ~12 h (Pro).
# Re-running this cell (or Cell 0) is all that is needed:
#   - Pipeline skips steps whose done_check file exists on Drive.
#   - Training auto-resumes from ckpt_latest (trainer crash-safe resume).
#   - At most save_every steps re-run (colab: 25 steps ~= 1-2 min).
#   - Eval steps pass --skip_existing so completed cells are skipped.
# -----------------------------------------------------------------------------
import os, subprocess, sys

# -- Re-mount Drive and re-clone if needed (idempotent) -----------------------
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
DRIVE_ROOT = "/content/drive/MyDrive/specdist"   # same as Cell 1 & 3

# Private repo: read GitHub token from Colab Secrets.
def _get_gh_token():
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t: return t
    except Exception: pass
    return os.environ.get("GITHUB_TOKEN", "")

_gh_tok    = _get_gh_token()
_clone_url = REPO_URL.replace("https://", f"https://{_gh_tok}@") if _gh_tok else REPO_URL

if not os.path.isdir(REPO_DIR):
    if not _gh_tok:
        print("⚠ GITHUB_TOKEN not set -- clone may fail for a private repo.")
    subprocess.run(["git", "clone", "--depth", "1", _clone_url, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", f"{GBV_DIR}/requirements.txt",
                    "bitsandbytes", "accelerate"], check=True)
    print("✓ Repo re-cloned, deps re-installed")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo still present -- updated")

os.chdir(GBV_DIR)

# -- Re-authenticate W&B -------------------------------------------------------
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        import wandb; wandb.login(key=key, relogin=True)
        print("✓ W&B re-authenticated")
except Exception:
    os.environ.setdefault("WANDB_MODE", "offline")
    print("⚠ W&B key not found -- logging offline")

# -- Re-set storage env vars ---------------------------------------------------
os.environ["SPECDIST_STORAGE_ROOT"] = DRIVE_ROOT
os.environ["SPECDIST_DB_PATH"]       = os.path.join(DRIVE_ROOT, "results.db")
os.environ["SPECDIST_LOGS_ROOT"]     = os.path.join(DRIVE_ROOT, "logs")

# -- Resume pipeline -----------------------------------------------------------
CONFIG = "colab_a100"   # must match what was used in Cell 3

print(f"\nResuming pipeline (config={CONFIG}, storage_root={DRIVE_ROOT})")
print("Pipeline will skip already-completed steps and resume from last failure.\n")

subprocess.run([
    sys.executable, "orchestration/experiment.py",
    "--config",       CONFIG,
    "--storage_root", DRIVE_ROOT,
    "--yes",
], cwd=GBV_DIR)

In [ ]:
# -----------------------------------------------------------------------------
# Cell 5 -- Monitor progress
#
# Safe to run any time, including while Cell 0 / Cell 3 is running.
# AUTO_REFRESH = True  -> re-prints every REFRESH_SECS seconds (live tail).
#                         Interrupt the cell (square button) to stop.
# AUTO_REFRESH = False -> single snapshot (default).
# -----------------------------------------------------------------------------
import os, json, glob, datetime, time
from IPython.display import clear_output

DRIVE_ROOT   = "/content/drive/MyDrive/specdist"
LOG_TAIL     = 60      # log lines to show
AUTO_REFRESH = False   # True = live tail loop
REFRESH_SECS = 20      # seconds between refreshes

LOG_FILE   = f"{DRIVE_ROOT}/logs/pipeline_output.log"
STATE_FILE = f"{DRIVE_ROOT}/pipeline_state_colab.json"

def _show():
    sep = "=" * 62
    # -- 1. Pipeline state ----------------------------------------------------
    print(sep)
    print("PIPELINE STATE")
    print(sep)
    if os.path.exists(STATE_FILE):
        state = json.load(open(STATE_FILE, encoding="utf-8"))
        steps  = state.get("steps", {})
        counts = {"done": 0, "running": 0, "pending": 0, "error": 0}
        for sid, info in steps.items():
            s    = info.get("status", "pending")
            counts[s] = counts.get(s, 0) + 1
            icon = {"done": "OK", "running": ">>", "error": "!!", "pending": ".."}.get(s, "..")
            ts   = (info.get("finished_at") or info.get("started_at") or "")[:16]
            print(f"  [{icon}] {sid:45s}  {s:8s}  {ts}")
        print(f"\n  Done: {counts['done']}  Running: {counts['running']}"
              f"  Pending: {counts.get('pending',0)}  Error: {counts['error']}")
    else:
        print(f"  State file not found: {STATE_FILE}")
        print("  Pipeline has not started yet (or Drive is not mounted).")

    # -- 2. Log tail ----------------------------------------------------------
    print()
    print(sep)
    print(f"PIPELINE LOG (last {LOG_TAIL} lines)")
    print(sep)
    if os.path.exists(LOG_FILE):
        lines = open(LOG_FILE, encoding="utf-8", errors="replace").readlines()
        print("".join(lines[-LOG_TAIL:]))
        age_s  = datetime.datetime.now().timestamp() - os.path.getmtime(LOG_FILE)
        status = "ACTIVE" if age_s < 120 else f"STALE ({int(age_s//60)} min ago)"
        print(f"[{len(lines)} lines | {status}]")
    else:
        print(f"  Not found: {LOG_FILE}")

    # -- 3. Error snapshots ---------------------------------------------------
    err_logs = sorted(glob.glob(f"{DRIVE_ROOT}/logs/step_*_error.log"))
    if err_logs:
        print()
        print(sep)
        print(f"ERROR SNAPSHOTS ({len(err_logs)} file(s))")
        print(sep)
        for f in err_logs:
            print(f"\n--- {os.path.basename(f)} ---")
            txt = open(f, encoding="utf-8", errors="replace").read()
            print(txt[-2000:] if len(txt) > 2000 else txt)

    if AUTO_REFRESH:
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"\n[{ts}  |  next refresh in {REFRESH_SECS}s  |  interrupt to stop]")

if AUTO_REFRESH:
    print(f"Auto-refresh every {REFRESH_SECS}s -- interrupt cell (square button) to stop.")
    while True:
        clear_output(wait=True)
        _show()
        time.sleep(REFRESH_SECS)
else:
    _show()

In [ ]:
# -----------------------------------------------------------------------------
# Cell 6 -- Live results dashboard
#
# Starts the Flask dashboard server bound to 0.0.0.0 so Colab's port proxy
# can tunnel it to a public HTTPS URL you can open in any browser tab.
#
# Safe to run while Cell 3 is training -- the dashboard reads the DB live
# and shows new eval rows as they land.
#
# Shows:
#   • Block efficiency heatmap (loss × verifier)
#   • Training loss curves
#   • Per-step acceptance-rate breakdown
#   • Live pipeline log tail
#
# To stop: Runtime -> Interrupt execution  (or just close the tab -- server
# keeps running in background until the Colab session dies)
# -----------------------------------------------------------------------------
import os, sys, time, threading, subprocess
from google.colab.output import eval_js

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"
PORT       = 5000

# Point dashboard at Drive DB + logs (not ephemeral /content/)
os.environ["SPECDIST_DB_PATH"]   = f"{DRIVE_ROOT}/results.db"
os.environ["SPECDIST_LOGS_ROOT"] = f"{DRIVE_ROOT}/logs"

if not os.path.exists(os.environ["SPECDIST_DB_PATH"]):
    print(f"⚠ No results DB yet at {os.environ['SPECDIST_DB_PATH']}")
    print("  Run Cell 3 first to generate some results, then come back here.")
else:
    # Launch Flask server in a background thread so this cell doesn't block
    _server_proc = [None]

    def _start_server():
        _server_proc[0] = subprocess.Popen(
            [
                sys.executable,
                f"{GBV_DIR}/dashboard/training_dashboard.py",
                "--host", "0.0.0.0",
                "--port", str(PORT),
            ],
            env=os.environ.copy(),
        )
        _server_proc[0].wait()

    t = threading.Thread(target=_start_server, daemon=True)
    t.start()

    # Give Flask a moment to bind the port
    time.sleep(3)

    # Get the Colab-proxied public URL for this port
    public_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")

    print(f"✓ Dashboard running")
    print(f"  Open this URL in any browser tab:")
    print(f"  {public_url}")
    print()
    print(f"  DB   : {os.environ['SPECDIST_DB_PATH']}")
    print(f"  Logs : {os.environ['SPECDIST_LOGS_ROOT']}")
    print(f"  Port : {PORT} (proxied by Colab to HTTPS above)")
    print()
    print("  The dashboard auto-refreshes every 15 s -- new eval rows appear as they land.")
    print("  To stop the server: Runtime -> Interrupt execution")

---

## Tree-Structured Losses (Track B) — Train One at a Time

Use **Cell 7** (single loss) or **Cell 8** (all losses) for Track B.
Tree losses use the three-phase on-policy loop: Phase A = sample K i.i.d.
draft paths from the student; Phase B = teacher scoring (no grad); Phase C =
student forward with grad + loss.backward().

| Loss | What it targets | Key notes |
|---|---|---|
| `kl_tree` | Forward KL at each tree node | Numerically stable; universal on-policy baseline |
| `rev_kl_tree` | Reverse KL at each node | Mode-seeking variant |
| `jsd_tree` | Symmetric JSD at each node | Bounded [0, log 2] |
| `bv_tree` | BV acceptance integral surrogate | Targets `bv_verify` directly |
| `gbv_tree` | GBV acceptance + q-skew | Targets `gbv_verify`; stable for K≤4 |
| `traversal_tree` | Traversal leaf-weight product | Targets `traversal_verify` |
| `ebe_tree` | On-policy EBE ablation | Same formula as flat EBE, but on-policy data |
| `online_kl_tree` | Online kl_tree (no replay buffer) | H6: online tree vs flat online |
| `online_ebe_tree` | Online ebe_tree (no replay buffer) | |

**Evaluation**: tree losses are evaluated with `bv`, `gbv`, `traversal` only —
OT verifiers (`specinfer`) are out-of-distribution for tree-trained models.

**Prerequisites**: Cells 1 + 2 must have run (Drive mounted, deps installed,
W&B authenticated). They remain valid across the same Colab session.


In [ ]:
# =============================================================================
# Cell 7 -- Train a single tree loss (A100)
#
# 1. Set LOSS to the tree loss you want (see table above).
# 2. Run this cell.  Runs:  train  ->  merge LoRA  ->  eval on GSM8K
# 3. Change LOSS and re-run for the next one.
#    Completed steps are always skipped (crash-safe).
#
# PROFILE: all training settings come from the YAML.
#   profiles/a100_tree_losses  8B teacher (BF16), 2000 steps, r=16  (default)
# =============================================================================

import os, subprocess, sys, threading, time

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Config (edit these) ──────────────────────────────────────────────────────
# Offline losses (7):  kl_tree | rev_kl_tree | jsd_tree
#                      bv_tree | gbv_tree | traversal_tree | ebe_tree
# Online variants (2): online_kl_tree | online_ebe_tree
LOSS       = "kl_tree"                     # <-- change this
PROFILE    = "profiles/a100_tree_losses"   # all settings come from this YAML
SMOKE      = False
BACKGROUND = False
# ─────────────────────────────────────────────────────────────────────────────

VALID = [
    "kl_tree", "rev_kl_tree", "jsd_tree",
    "bv_tree", "gbv_tree", "traversal_tree", "ebe_tree",
    "online_kl_tree", "online_ebe_tree",
]
if LOSS not in VALID:
    raise ValueError(f"Unknown loss {LOSS!r}. Valid: {VALID}")
if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("Drive not mounted -- run Cell 1 first.")
if not os.path.isdir(GBV_DIR):
    raise RuntimeError("Repo not found -- run Cell 1 first.")

os.chdir(GBV_DIR)
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      f"{DRIVE_ROOT}/results.db",
    "SPECDIST_LOGS_ROOT":    f"{DRIVE_ROOT}/logs",
})
LOG_FILE = f"{DRIVE_ROOT}/logs/tree_{LOSS}_output.log"

# Show what the YAML profile will use (reads the actual YAML file)
def _show_profile(profile, gbv_dir):
    """Read the YAML profile and print key params so users know what will run."""
    import yaml, os
    yaml_path = os.path.join(gbv_dir, "orchestration", "configs",
                             profile.replace("/", os.sep) + ".yaml")
    try:
        with open(yaml_path, encoding="utf-8") as _f:
            cfg = yaml.safe_load(_f)
        tr   = cfg.get("training", {})
        hw   = cfg.get("hardware", {})
        mdl  = cfg.get("models", {})
        tree = cfg.get("tree_training", {})
        ev   = cfg.get("evaluation", {})
        print(f"  Profile  : {profile}")
        print(f"  Teacher  : {mdl.get('target', '?')}  "
              f"({'4-bit NF4' if hw.get('load_in_4bit') else 'BF16'})")
        print(f"  Draft    : {mdl.get('draft', '?')}")
        print(f"  Steps    : {tr.get('steps', '?')}  "
              f"lr={tr.get('lr', '?')}  lora_r={tr.get('lora_r', '?')}")
        print(f"  Tree     : K={tree.get('tree_K', '?')}  L={tree.get('tree_L', '?')}")
        print(f"  Eval     : modes={ev.get('modes', '?')}  "
              f"n_prompts={ev.get('n_prompts', '?')}")
    except Exception as _e:
        print(f"  Profile  : {profile}  (could not read YAML: {_e})")
_show_profile(PROFILE, GBV_DIR)
print(f"  Loss     : {LOSS}{'  (SMOKE: 10 steps)' if SMOKE else ''}")
print(f"  Log      : {LOG_FILE}")

import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"  GPU      : {torch.cuda.get_device_name(0)}  "
          f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
    if total / 1024**3 < 30:
        print("  NOTE: Non-A100 GPU detected. The a100_tree_losses profile uses")
        print("        Qwen3-8B BF16 (~16 GB). On T4 use colab_tree_losses instead.")

cmd = [
    sys.executable, "orchestration/experiment.py",
    "--config",       PROFILE,
    "--losses",       LOSS,
    "--storage_root", DRIVE_ROOT,
    "--yes",
]
if SMOKE:
    cmd.append("--smoke")

if BACKGROUND:
    log_fh = open(LOG_FILE, "a", buffering=1)
    _proc  = subprocess.Popen(cmd, cwd=GBV_DIR,
                               stdout=log_fh, stderr=subprocess.STDOUT)

    def _tail():
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as lf:
            lf.seek(0, 2)
            while _proc.poll() is None:
                line = lf.readline()
                if line: sys.stdout.write(line); sys.stdout.flush()
                else:    time.sleep(0.4)
            for line in lf: sys.stdout.write(line); sys.stdout.flush()

    threading.Thread(target=_tail, daemon=True).start()
    print(f"\nBackground PID {_proc.pid}")
    print("Monitor: run Cell 5 with AUTO_REFRESH=True")
    print(f"Stop:    import os, signal; os.kill({_proc.pid}, signal.SIGTERM)")
else:
    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print(f"\n[DONE] {LOSS} -- change LOSS above and re-run for the next one.")
        print("  Run Cell 6 to view results in the dashboard.")
    else:
        print(f"\n[FAIL] {LOSS} exited {result.returncode}")
        print("  Re-run this cell to resume from the last checkpoint.")
        print(f"  Full log: {LOG_FILE}")


In [ ]:
# =============================================================================
# Cell 8 -- Train ALL tree losses sequentially (A100)
#
# Runs all 9 tree losses one after another, stopping on first failure.
# Re-run to resume -- completed steps are always skipped.
#
# All training settings come from the YAML profile (PROFILE below).
# Edit the YAML to change steps, lr, lora_r, etc.  Do not duplicate
# those values here.
# =============================================================================

import os, subprocess, sys

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Config (edit these) ──────────────────────────────────────────────────────
PROFILE = "profiles/a100_tree_losses"  # all training settings come from this YAML
LOSSES_TO_RUN = [
    # Offline tree losses (run first)
    "kl_tree",         # H5 primary comparison point
    "rev_kl_tree",
    "jsd_tree",
    "bv_tree",
    "gbv_tree",        # stable at K<=4
    "traversal_tree",
    "ebe_tree",        # off-policy mismatch ablation
    # Online tree variants (run last)
    "online_kl_tree",  # H6 test
    "online_ebe_tree",
]
SMOKE = False  # True = 10-step crash check per loss
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("Drive not mounted -- run Cell 1 first.")

os.chdir(GBV_DIR)
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      f"{DRIVE_ROOT}/results.db",
    "SPECDIST_LOGS_ROOT":    f"{DRIVE_ROOT}/logs",
})

# Show what the YAML profile will use
def _show_profile(profile, gbv_dir):
    """Read the YAML profile and print key params so users know what will run."""
    import yaml, os
    yaml_path = os.path.join(gbv_dir, "orchestration", "configs",
                             profile.replace("/", os.sep) + ".yaml")
    try:
        with open(yaml_path, encoding="utf-8") as _f:
            cfg = yaml.safe_load(_f)
        tr   = cfg.get("training", {})
        hw   = cfg.get("hardware", {})
        mdl  = cfg.get("models", {})
        tree = cfg.get("tree_training", {})
        ev   = cfg.get("evaluation", {})
        print(f"  Profile  : {profile}")
        print(f"  Teacher  : {mdl.get('target', '?')}  "
              f"({'4-bit NF4' if hw.get('load_in_4bit') else 'BF16'})")
        print(f"  Draft    : {mdl.get('draft', '?')}")
        print(f"  Steps    : {tr.get('steps', '?')}  "
              f"lr={tr.get('lr', '?')}  lora_r={tr.get('lora_r', '?')}")
        print(f"  Tree     : K={tree.get('tree_K', '?')}  L={tree.get('tree_L', '?')}")
        print(f"  Eval     : modes={ev.get('modes', '?')}  "
              f"n_prompts={ev.get('n_prompts', '?')}")
    except Exception as _e:
        print(f"  Profile  : {profile}  (could not read YAML: {_e})")
_show_profile(PROFILE, GBV_DIR)
print(f"  Losses   : {len(LOSSES_TO_RUN)} tree losses{'  (SMOKE: 10 steps each)' if SMOKE else ''}")
print(f"  Resume   : completed steps are skipped on re-run")
print()

failed = False
for i, loss_name in enumerate(LOSSES_TO_RUN):
    print(f"{'='*60}")
    print(f"[{i+1}/{len(LOSSES_TO_RUN)}] {loss_name}")
    print(f"{'='*60}")

    cmd = [
        sys.executable, "orchestration/experiment.py",
        "--config",       PROFILE,
        "--losses",       loss_name,
        "--storage_root", DRIVE_ROOT,
        "--yes",
    ]
    if SMOKE:
        cmd.append("--smoke")

    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print(f"  [DONE] {loss_name}")
    else:
        print(f"  [FAIL] {loss_name} exited {result.returncode}")
        print("  Re-run this cell to resume (completed steps will be skipped).")
        failed = True
        break

if not failed:
    print("\n" + "="*60)
    print("All tree losses complete!")
    print(f"  DB  : {DRIVE_ROOT}/results.db")
    print("  Run Cell 6 for the dashboard (Ph 6 Tree Training tab).")
